## Generalized Linear Model

### Likelihood (transconjugants)

For each experiment \(i\), replica \(j\), within-replica observation \(k\), and condition \(c\),

$$
T_{i,j,k,c}\mid \lambda_{i,j,c},\,X_{i,c},\,\rho_k
\;\overset{\text{ind}}{\sim}\;
\text{Poisson}\!\big(\rho_k\,\lambda_{i,j,c}\big),
$$

with

$$
\lambda_{i,j,c}=\alpha_i\,\theta_{i,j,c}.
$$

### Indices and variables

- \(i \in \{1,\dots,10\}\): experiment  
- \(j \in \{A,B,C\}\): experiment replica  
- \(k \in \{1,\dots,9\}\): observations within each replica  
- \(c \in \{-1,1\}\): control (\(c=-1\)) or treatment (\(c=+1\))  
- \(T_{i,j,k,c}\): transconjugant count  
- \(X_{i,c}\): covariate vector (Temp, \([\mathrm{IBU}]\), \([\mathrm{DMSO}]\)) for experiment \(i\) under condition \(c\)  
- \(\rho_k = 10^{-\lceil k/3\rceil}\): dilution factor for transconjugant measurements  
- \(\alpha_i\): donor count (exposure) for experiment \(i\)  
- \(\theta_{i,j,c}\): transconjugation rate per donor  

---

## Linear predictor (log link)

We model \(\lambda_{i,j,c}\) using a log link with an exposure offset:

$$
\log(\lambda_{i,j,c})
=
\log(\alpha_i)
+ b_{i,j}
+ \boldsymbol{\beta}^{\top}F(\mathbf{x}_{i,c}).
$$

- \(\mathbf{x}_{i,c}\): raw covariates for experiment \(i\), condition \(c\)  
- \(F(\mathbf{x}_{i,c})\): feature map (e.g., main effects, interactions, transformations)  
- \(\boldsymbol{\beta}\in\mathbb{R}^p\): fixed-effect coefficients  

### Random effect (replica-level)

$$
b_{i,j}\mid \sigma_b^2
\;\overset{\text{iid}}{\sim}\;
\mathcal{N}(0,\sigma_b^2).
$$

---

## Bayesian model for donors

### Likelihood (donor measurements)

For donor count measurements indexed by \(k\),

$$
D_{i,k}\mid \alpha_i,\,\delta_k
\;\overset{\text{ind}}{\sim}\;
\text{Poisson}\!\big(\delta_k\,\alpha_i\big),
$$

where

- \(D_{i,k}\): donor count measurement for experiment \(i\)  
- \(\delta_k = 10^{-5-\lceil k/3\rceil}\): dilution factor for donor measurements  

---

## Priors

### Donor-level prior

$$
\log(\alpha_i)\mid \mu_\alpha,\,\sigma_\alpha^2
\;\overset{\text{iid}}{\sim}\;
\mathcal{N}(\mu_\alpha,\sigma_\alpha^2).
$$

### Hyperpriors for donor parameters

$$
\mu_\alpha \sim \mathcal{N}(\mu_{\alpha_0},\sigma_{\alpha_0}^2),
\qquad
\sigma_\alpha \sim \text{half-}\mathcal{N}(0,s_{\alpha_0}).
$$

### Priors for fixed effects

$$
\boldsymbol{\beta}\mid \tau
\sim
\mathcal{N}\!\big(\mathbf{0},\,\tau^2 I_p\big),
\qquad
\tau \sim \text{half-}\mathcal{N}(0,1).
$$

### Prior for random-effect variance

$$
\sigma_b^2 \sim \text{Inverse-Gamma}(2,1).
$$

In [1]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
from numpy.random import default_rng
import os
import arviz as az

# Import functions
from cmdstanpy import CmdStanModel
from tensorflow_probability.substrates import numpy as tfp
tfd = tfp.distributions



import os
from pathlib import Path

# Find and navigate to Progetto_Bayesian_Statistics
current = Path.cwd()

# Check if we're already there
if current.name == 'Progetto_Bayesian_Statistics':
    print(f"Already in project root: {current}")
else:
    # Search up to parent directories
    for parent in [current] + list(current.parents):
        if parent.name == 'Progetto_Bayesian_Statistics':
            os.chdir(parent)
            print(f"Changed to: {parent}")
            break
    else:
        raise FileNotFoundError("Progetto_Bayesian_Statistics folder not found")

# Verify
assert Path('src').exists(), "src/ folder not found"
print(f"Current directory: {os.getcwd()}")
print(f"Contents: {os.listdir()}")


# Create ./stan folder if does not exists
STAN_PATH = "./src/stan_models/"
import cmdstanpy
print(cmdstanpy.cmdstan_path())

/home/ahbagheri/Projects/envs/.ML/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Changed to: /home/ahbagheri/Projects/Bayesian/Progetto_Bayesian_Statistics
Current directory: /home/ahbagheri/Projects/Bayesian/Progetto_Bayesian_Statistics
Contents: ['.git', '.gitignore', 'requirements.bk.txt', 'README.md', 'data', 'requirements.txt', 'materials', 'src']
/home/ahbagheri/.cmdstan/cmdstan-2.38.0


In [2]:
#loading data
#get the data
data=pd.read_csv("./data/data.csv")
data_donors=pd.read_csv("./data/data_donatori.csv")
data=data[data['Conta']!='TMTC']
data_donors=data_donors[data_donors['Conta']!='TMTC']
#Rename columns
data.rename(columns={'Idx Replica Condizione': 'Idx Replica Experiment','Controllo':'Control','Idx Replica Diluizione':'Idx Replica Diluition','Conta':'Count'}, inplace=True)
data_donors.rename(columns={'Idx Replica Diluizione':'Idx Replica Experiment','Conta':'Count'}, inplace=True)

#Modify values in some columns:
#Modify Idx Experiment such that experiments 5 and 6 have the same idx, experiments 7 and 8, experiment 5,6 has the same index as 5 and 6 and experiment 7,8 the same as 7 and 8
#but first convert all to string to avoid problems with mixed types
data['Idx Experiment'] = data['Idx Experiment'].astype(str)
data['Idx Experiment'] = data['Idx Experiment'].replace({'6': '5', '7':'6','8': '6', '9':'7','10':'8','5,6': '5', '7,8':'6'})
#now reconvert to numeric
data['Idx Experiment'] = pd.to_numeric(data['Idx Experiment'])

data_donors['Idx Experiment'] = data_donors['Idx Experiment'].astype(str)
data_donors['Idx Experiment'] = data_donors['Idx Experiment'].replace({'6': '5', '7':'6','8': '6', '9':'7','10':'8','5,6': '5', '7,8':'6'})

#modify control column: convert 'Yes' to 1 and 'No' to 0
data['Control'] = data['Control'].replace({'Yes': 1, 'No': 0})



#now reconvert to numeric
data_donors['Idx Experiment'] = pd.to_numeric(data_donors['Idx Experiment'])

print(data['Idx Experiment'].unique())
print(data_donors['Idx Experiment'].unique())

#convert to numeric all the others columns I need 
#convert A;B;C in 1;2;3
data['Idx Replica Experiment'] = data['Idx Replica Experiment'].astype(str)
data['Idx Replica Experiment'] = data['Idx Replica Experiment'].replace({'A': '1', 'B':'2','C': '3'})
#now reconvert to numeric
data['Idx Replica Experiment'] = pd.to_numeric(data['Idx Replica Experiment'])

data['Count'] = pd.to_numeric(data['Count'])
data['Idx Replica Diluition'] = pd.to_numeric(data['Idx Replica Diluition'])
data['Diluition'] = pd.to_numeric(data['Diluition'])
data['IBU'] = pd.to_numeric(data['IBU'])
data['DMSO'] = pd.to_numeric(data['DMSO'])
data['Temp'] = pd.to_numeric(data['Temp'])
data_donors['Count'] = pd.to_numeric(data_donors['Count'])
data_donors['Idx Replica Experiment'] = pd.to_numeric(data_donors['Idx Replica Experiment'])  

[1 2 3 4 5 6 7 8]
[1 2 3 4 5 6 7 8]


/tmp/ipykernel_2798/3506374784.py:23: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  data['Control'] = data['Control'].replace({'Yes': 1, 'No': 0})


In [3]:
#Extract the data we need for the Stan model 
#from trasconjugant dataset
Y=data.Count.values
N=len(Y)
idx_experiment=data['Idx Experiment'].values
I=len(np.unique(idx_experiment))
idx_experiment_replica=data['Idx Replica Experiment'].values
J=len(np.unique(idx_experiment_replica))
dil_trasc=data['Diluition'].values
rho_trasc = np.power(10.0, dil_trasc)


#from donor dataset
D=data_donors.Count.values
M=len(D)
idx_donor_experiment=data_donors['Idx Experiment'].values
dil_donor=data_donors['Diluition'].values
rho_donor=np.power(10.0, dil_donor)


#build design matrix X
IBU=data['IBU'].values
DMSO=data['DMSO'].values
Temp=data['Temp'].values

#normalize covariates
IBU=(IBU - np.mean(IBU)) / np.std(IBU)
DMSO=(DMSO - np.mean(DMSO)) / np.std(DMSO)
Temp=(Temp - np.mean(Temp)) / np.std(Temp)  

X = np.column_stack([np.ones_like(Y),IBU,DMSO,Temp])
model_names = {"ab_baseline_simple_poi.stan": {}, "ab_regularized_horseshoe.stan": {"p0":2}, "ab_lasso.stan": {}, "ab_horseshoe.stan": {}, "ab_r2d2.stan": {"R2_mean": 0.5,"R2_prec": 2.0 }}
models = {}
for key in model_names:
    print(f"Compiling model: {key}")
    glm = CmdStanModel(stan_file=f"{STAN_PATH}/{key}")
    models[key] = {"model": glm}

Compiling model: ab_baseline_simple_poi.stan
Compiling model: ab_regularized_horseshoe.stan
Compiling model: ab_lasso.stan
Compiling model: ab_horseshoe.stan
Compiling model: ab_r2d2.stan


In [4]:
# Input data
glm_data = {
    "N": N,
    "M": M,
    "I":I,
    "J":J,
    "p": X.shape[1],
    "Y": Y,
    "D": D,
    "X": X,
    "rho_trasc": rho_trasc,
    "rho_donor": rho_donor,
    "idx_experiment": idx_experiment,
    "idx_experiment_replica": idx_experiment_replica,
    "idx_donor_experiment": idx_donor_experiment
}


for key in model_names:
    new_data = glm_data.copy()
    new_data.update(model_names[key])
    print(f"Compiling model: {key}")
    models[key]["data"] = new_data

# # sample from all:
# for key in models:
#     print(f"Sampling model: {key}")
#     glm_fit = models[key]["model"].sample(data=models[key]["data"], chains=4, parallel_chains=4, 
#                                  iter_warmup=1000, iter_sampling=5000,show_console=False)
#     models[key]["fit"] = glm_fit

# for key in models:
#     models[key]["az"] = az.from_cmdstanpy(models[key]["fit"])

Compiling model: ab_baseline_simple_poi.stan
Compiling model: ab_regularized_horseshoe.stan
Compiling model: ab_lasso.stan
Compiling model: ab_horseshoe.stan
Compiling model: ab_r2d2.stan


In [5]:

from concurrent.futures import ProcessPoolExecutor, as_completed

def sample_model(key, model_config):
    glm_fit = model_config["model"].sample(
        data=model_config["data"], 
        chains=4, 
        parallel_chains=4,
        iter_warmup=1000, 
        iter_sampling=5000,
        show_console=False
    )
    return key, glm_fit

# Parallel sampling with worker limit
max_workers = 4  # adjust based on your RAM/CPU
with ProcessPoolExecutor(max_workers=max_workers) as executor:
    futures = {executor.submit(sample_model, k, v): k for k, v in models.items()}
    for future in as_completed(futures):
        key, fit = future.result()
        models[key]["fit"] = fit
        models[key]["az"] = az.from_cmdstanpy(fit)
        print(f"Completed: {key}")

11:09:31 - cmdstanpy - INFO - CmdStan start processing
11:09:31 - cmdstanpy - INFO - CmdStan start processing
chain 1:   0%|          | 0/6000 [00:00<?, ?it/s, (Warmup)]

11:09:31 - cmdstanpy - INFO - CmdStan start processing

chain 1:   0%|          | 0/6000 [00:00<?, ?it/s, (Warmup)]

chain 2:   0%|          | 0/6000 [00:00<?, ?it/s, (Warmup)]



chain 4:   0%|          | 0/6000 [00:00<?, ?it/s, (Warmup)]





chain 1:   2%|▏         | 100/6000 [00:00<00:56, 104.91it/s, (Warmup)]






chain 1:   3%|▎         | 200/6000 [00:01<00:49, 117.23it/s, (Warmup)]

chain 1:   2%|▏         | 100/6000 [00:02<02:01, 48.55it/s, (Warmup)]









chain 1:   3%|▎         | 200/6000 [00:03<01:23, 69.66it/s, (Warmup)]


chain 1:   7%|▋         | 400/6000 [00:03<00:44, 125.95it/s, (Warmup)]


chain 1:   7%|▋         | 400/6000 [00:03<00:47, 118.27it/s, (Warmup)]

chain 1:   5%|▌         | 300/6000 [00:03<01:06, 85.52it/s, (Warmup)] 


chain 4:   8%|▊         | 500/6000 [00:03<00:39, 137.68it/s, (Warm

11:10:25 - cmdstanpy - INFO - CmdStan done processing.
11:10:25 - cmdstanpy - WARNING - Non-fatal error during sampling:
Exception: poisson_lpmf: Rate parameter[13] is -nan, but must be nonnegative! (in 'ab_horseshoe.stan', line 43, column 4 to column 33)
Exception: poisson_lpmf: Rate parameter[73] is -nan, but must be nonnegative! (in 'ab_horseshoe.stan', line 43, column 4 to column 33)
Exception: poisson_lpmf: Rate parameter[40] is -nan, but must be nonnegative! (in 'ab_horseshoe.stan', line 43, column 4 to column 33)
Consider re-running with show_console=True if the above output is unclear!


chain 1:  92%|█████████▏| 5500/6000 [00:54<00:04, 105.20it/s, (Sampling)]


chain 1:  87%|████████▋ | 5200/6000 [00:54<00:09, 88.19it/s, (Sampling)]


chain 1:  93%|█████████▎| 5600/6000 [00:55<00:03, 110.09it/s, (Sampling)]


11:10:26 - cmdstanpy - WARNING - Some chains may have failed to converge.
	Chain 1 had 111 divergent transitions (2.2%)
	Chain 2 had 85 divergent transitions (1.7%)
	Chai

Completed: ab_horseshoe.stan


chain 2: 100%|██████████| 6000/6000 [00:58<00:00, 102.49it/s, (Sampling completed)]

chain 3: 100%|██████████| 6000/6000 [00:58<00:00, 102.48it/s, (Sampling completed)]


chain 4: 100%|██████████| 6000/6000 [00:58<00:00, 102.45it/s, (Sampling completed)]


11:10:29 - cmdstanpy - INFO - CmdStan done processing.
11:10:29 - cmdstanpy - WARNING - Non-fatal error during sampling:
Exception: normal_lpdf: Scale parameter is 0, but must be positive! (in 'ab_lasso.stan', line 46, column 4 to column 43)
Exception: double_exponential_lpdf: Scale parameter is inf, but must be positive finite! (in 'ab_lasso.stan', line 52, column 4 to column 53)
Consider re-running with show_console=True if the above output is unclear!


chain 2: 100%|██████████| 6000/6000 [00:59<00:00, 100.95it/s, (Sampling completed)]

chain 3: 100%|██████████| 6000/6000 [00:59<00:00, 100.94it/s, (Sampling completed)]


chain 4: 100%|██████████| 6000/6000 [00:59<00:00, 100.92it/s, (Sampling completed)]


...

chain 1:  10%|█         | 600/6000 [00:06<00:47, 114.33it/s, (Warmup)]


chain 1:  12%|█▏        | 700/6000 [00:07<00:40, 129.40it/s, (Warmup)]


chain 1:  13%|█▎        | 800/6000 [00:07<00:36, 143.43it/s, (Warmup)]


chain 1:  17%|█▋        | 1000/6000 [00:08<00:30, 162.61it/s, (Sampling)]






chain 3:  17%|█▋        | 1000/6000 [00:09<00:32, 155.73it/s, (Warmup)]


Completed: ab_baseline_simple_poi.stan


chain 1:  20%|██        | 1200/6000 [00:10<00:31, 150.56it/s, (Sampling)]




chain 1:  22%|██▏       | 1300/6000 [00:10<00:32, 144.10it/s, (Sampling)]

chain 1:  23%|██▎       | 1400/6000 [00:11<00:32, 141.79it/s, (Sampling)]


chain 1:  25%|██▌       | 1500/6000 [00:12<00:32, 140.46it/s, (Sampling)]


chain 1:  27%|██▋       | 1600/6000 [00:12<00:29, 150.58it/s, (Sampling)]


chain 1:  28%|██▊       | 1700/6000 [00:13<00:27, 155.72it/s, (Sampling)]


chain 1:  30%|███       | 1800/6000 [00:14<00:26, 157.99it/s, (Sampling)]


chain 1:  32%|███▏      | 1900/6000 [00:14<00:26, 153.66it/s, (Sampling)]


chain 1:  33%|███▎      | 2000/6000 [00:15<00:25, 158.12it/s, (Sampling)]


chain 1:  35%|███▌      | 2100/6000 [00:15<00:23, 164.95it/s, (Sampling)]



chain 1:  37%|███▋      | 2200/6000 [00:16<00:22, 168.21it/s, (Sampling)]


chain 1:  38%|███▊      | 2300/6000 [00:17<00:21, 169.38it/s, (Sampling)]


chain 1:  40%|████      | 2400/6000 [00:17<00:20, 175.72it/s, (Sampling)]


chain 1:  

11:11:06 - cmdstanpy - INFO - CmdStan done processing.
11:11:06 - cmdstanpy - WARNING - Non-fatal error during sampling:
Exception: normal_lpdf: Scale parameter is 0, but must be positive! (in 'ab_r2d2.stan', line 59, column 4 to column 43)
Exception: normal_lpdf: Scale parameter is 0, but must be positive! (in 'ab_r2d2.stan', line 72, column 4 to column 51)
Exception: normal_lpdf: Scale parameter is 0, but must be positive! (in 'ab_r2d2.stan', line 72, column 4 to column 51)
Consider re-running with show_console=True if the above output is unclear!


Completed: ab_r2d2.stan


In [ ]:

for key in models:
    poi_glm_data = models[key]["az"]
    print("divergence of ",key,np.sum(poi_glm_data.sample_stats.diverging))

divergence of  ab_baseline_simple_poi.stan <xarray.DataArray 'diverging' ()> Size: 8B
array(0)
divergence of  ab_regularized_horseshoe.stan <xarray.DataArray 'diverging' ()> Size: 8B
array(367)
divergence of  ab_lasso.stan <xarray.DataArray 'diverging' ()> Size: 8B
array(0)
divergence of  ab_horseshoe.stan <xarray.DataArray 'diverging' ()> Size: 8B
array(343)
divergence of  ab_r2d2.stan <xarray.DataArray 'diverging' ()> Size: 8B
array(0)
